# 02 · Camada Silver — Transformação (T) e SCD Tipo 2

A camada silver recebe os dados do bronze e aplica **limpeza, normalização e enriquecimento**, produzindo tabelas prontas para a camada gold.

**Transformações aplicadas:**
- Resolução de NULL com valores padrão (brand → 'Unknown', size/color → 'N/A').
- JOINs para desnormalizar descrições (colorname, stockgroupname).
- Trim em colunas de texto.
- Conversão de `isfinalized` (boolean → SMALLINT 0/1).
- **SCD Tipo 2** na `dim_stockitems_scd2`: historial completo de alterações com `valid_from`, `valid_to` e `is_current`.

## 1. Imports e ligações

In [ ]:
import pandas as pd
from datetime import datetime, date, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

engine_dwh = create_engine(
    f"postgresql+psycopg2://{os.getenv('DWH_USER')}:{os.getenv('DWH_PASSWORD')}@"
    f"{os.getenv('DWH_HOST')}:{os.getenv('DWH_PORT')}/{os.getenv('DWH_DB')}"
)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)
SCD2_SENTINEL = date(9999, 12, 31)

print(f"✓ Engine criado. Início: {run_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")

## 2. Preparação das tabelas Silver

In [ ]:
DDL_SCD2 = """
    CREATE TABLE IF NOT EXISTS silver.dim_stockitems_scd2 (
        scd_key            SERIAL PRIMARY KEY,
        stockitemid        INT NOT NULL,
        stockitemname      VARCHAR(255) NOT NULL,
        brand              VARCHAR(100),
        colorname          VARCHAR(50),
        size               VARCHAR(50),
        leadtimedays       INT,
        quantityperouter   INT,
        stockgroupname     VARCHAR(100),
        valid_from         DATE NOT NULL,
        valid_to           DATE NOT NULL DEFAULT '9999-12-31',
        is_current         BOOLEAN NOT NULL DEFAULT TRUE,
        _source_snapshot   INT NOT NULL,
        _loaded_at         TIMESTAMP NOT NULL
    );
"""

TRUNCATE_STAGING = """
    TRUNCATE TABLE 
        silver.stg_stockitems, silver.stg_customers, silver.stg_geography, 
        silver.stg_salespersons, silver.stg_invoices, silver.stg_invoicelines, 
        silver.stg_orders, silver.stg_orderlines, silver.stg_customertransactions;
"""

with engine_dwh.begin() as conn:
    conn.execute(text(DDL_SCD2))
    conn.execute(text(TRUNCATE_STAGING))

print("✓ Tabelas criadas/truncadas (SCD2 preservada).")

## 3. Funções auxiliares

In [ ]:
def read_active_snapshot(table_name: str) -> pd.DataFrame:
    """Lê o estado ativo do último snapshot de uma tabela bronze."""
    sql = f"""
        SELECT * FROM bronze.{table_name}
        WHERE _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name})
          AND _change_op != 'DELETE'
    """
    try:
        with engine_dwh.connect() as conn:
            return pd.read_sql(text(sql), conn)
    except:
        return pd.DataFrame()

def clean_str(s: pd.Series) -> pd.Series:
    return s.str.strip().replace("", None)

print("✓ Funções definidas.")

## 4. `stg_stockitems` + SCD Tipo 2

In [ ]:
SCD2_COLS = ["stockitemname", "brand", "colorname", "size", "stockgroupname", "leadtimedays", "quantityperouter"]

df_items  = read_active_snapshot("stockitems")
df_colors = read_active_snapshot("colors")
df_sisg   = read_active_snapshot("stockitemstockgroups")
df_sg     = read_active_snapshot("stockgroups")

# BUG FIX: Proteger contra bronze vazio para evitar ValueError em snap_id
if df_items.empty or df_items["_snapshot_id"].isna().all():
    print("⚠ bronze.stockitems está vazia ou sem snapshot válido — secção ignorada.")
else:
    snap_id = int(df_items["_snapshot_id"].max())

    # --- Transformações Standard ---
    df = df_items.merge(df_colors[["colorid", "colorname"]], on="colorid", how="left")
    df_grp = df_sisg.merge(df_sg[["stockgroupid", "stockgroupname"]], on="stockgroupid", how="left").drop_duplicates("stockitemid")
    df = df.merge(df_grp[["stockitemid", "stockgroupname"]], on="stockitemid", how="left")

    df["brand"] = df["brand"].fillna("Unknown").pipe(clean_str)
    df["colorname"] = df["colorname"].fillna("N/A").pipe(clean_str)
    df["size"] = df["size"].fillna("N/A").pipe(clean_str)
    df["stockgroupname"] = df["stockgroupname"].fillna("N/A").pipe(clean_str)
    df["stockitemname"] = clean_str(df["stockitemname"])

    # --- Staging (Recarga Completa) ---
    silver_items = df[["stockitemid", "stockitemname", "brand", "colorname", "size", "leadtimedays", "quantityperouter", "stockgroupname"]].copy()
    silver_items["_loaded_at"], silver_items["_source_snapshot"] = run_at, snap_id
    silver_items.to_sql("stg_stockitems", engine_dwh, schema="silver", if_exists="append", index=False)

    # --- SCD Tipo 2 (Histórico Acumulado) ---
    today = run_at.date()
    yesterday = date.fromordinal(today.toordinal() - 1)

    with engine_dwh.connect() as conn:
        df_current = pd.read_sql(text("SELECT * FROM silver.dim_stockitems_scd2 WHERE is_current = TRUE"), conn)

    if df_current.empty:
        # Carga Inicial Total
        to_insert = silver_items.copy()
        to_insert["valid_from"], to_insert["valid_to"], to_insert["is_current"] = today, SCD2_SENTINEL, True
        to_insert.to_sql("dim_stockitems_scd2", engine_dwh, schema="silver", if_exists="append", index=False)
        print(f"✓ SCD2: Carga inicial concluída ({len(to_insert)} registos).")
    else:
        # Comparação e Deteção de Alterações
        merged = silver_items.merge(df_current, on="stockitemid", how="left", suffixes=('_new', '_old'))
        
        # 1. Items Novos
        new_mask = merged['is_current'].isna()
        
        # 2. Items Alterados (comparamos campos SCD2)
        changed_mask = pd.Series(False, index=merged.index)
        for col in SCD2_COLS:
             changed_mask |= (merged[f"{col}_new"].fillna("").astype(str) != merged[f"{col}_old"].fillna("").astype(str))
        changed_mask &= ~new_mask

        # Ações na Base de Dados
        if changed_mask.any():
            scd_keys_to_expire = merged.loc[changed_mask, "scd_key"].tolist()
            with engine_dwh.begin() as conn:
                conn.execute(text("UPDATE silver.dim_stockitems_scd2 SET valid_to = :vt, is_current = FALSE WHERE scd_key = ANY(:keys)"), 
                             {"vt": yesterday, "keys": scd_keys_to_expire})
            
        # Inserir novos e novas versões
        to_insert = merged[new_mask | changed_mask].copy()
        if not to_insert.empty:
            # Renomear colunas _new de volta para o original
            final_insert = pd.DataFrame()
            for c in SCD2_COLS + ["stockitemid"]: final_insert[c] = to_insert[f"{c}_new"]
            final_insert["valid_from"], final_insert["valid_to"], final_insert["is_current"] = today, SCD2_SENTINEL, True
            final_insert["_source_snapshot"], final_insert["_loaded_at"] = snap_id, run_at
            final_insert.to_sql("dim_stockitems_scd2", engine_dwh, schema="silver", if_exists="append", index=False)

        print(f"✓ SCD2 processado: {new_mask.sum()} novos, {changed_mask.sum()} alterados.")

## 5. `stg_customers`

In [ ]:
df_cust = read_active_snapshot("customers")
df_bg   = read_active_snapshot("buyinggroups")
df_cc   = read_active_snapshot("customercategories")

if not df_cust.empty:
    snap_id = int(df_cust["_snapshot_id"].max())
    df = df_cust.merge(df_bg[["buyinggroupid", "buyinggroupname"]], on="buyinggroupid", how="left")
    df = df.merge(df_cc[["customercategoryid", "customercategoryname"]], on="customercategoryid", how="left")

    df["buyinggroupname"] = df["buyinggroupname"].fillna("Sem Grupo").pipe(clean_str)
    df["customername"] = clean_str(df["customername"])
    df["creditlimit"] = df["creditlimit"].fillna(0.00)

    silver_cust = df[["customerid", "customername", "buyinggroupname", "customercategoryname", "creditlimit", "accountopeneddate", "paymentdays", "deliverycityid"]].copy()
    silver_cust["_loaded_at"], silver_cust["_source_snapshot"] = run_at, snap_id
    silver_cust.to_sql("stg_customers", engine_dwh, schema="silver", if_exists="append", index=False)
    print(f"✓ stg_customers processado ({len(silver_cust)} registos).")

## 6. `stg_customertransactions`

In [ ]:
with engine_dwh.connect() as conn:
    df_ct = pd.read_sql(text("SELECT * FROM bronze.customertransactions"), conn)

if not df_ct.empty:
    df_ct["isfinalized"] = df_ct["isfinalized"].fillna(False).astype(int) # Conversão Boolean para SMALLINT
    silver_ct = df_ct[["customertransactionid", "customerid", "transactiondate", "transactionamount", "outstandingbalance", "isfinalized"]].copy()
    silver_ct["_loaded_at"] = run_at
    silver_ct.to_sql("stg_customertransactions", engine_dwh, schema="silver", if_exists="append", index=False)
    print(f"✓ stg_customertransactions processado.")